
# Taxi Zones Silver Layer Data Quality Validation

**Purpose:** Validate the Silver Taxi Zones table after transformation from Bronze.

**Source:** `ftw-week-08`.`02-bronze`.taxi_zones_raw

**Target:** `ftw-week-08`.`03-silver`.taxi_zones_clean

**DQ Results:** `ftw-week-08`.`01-control`.taxi_zones_silver_dq_results

### Validation areas
- Required fields
- LocationID validity and uniqueness
- Silver transformations
- Zone classification rules
- Bronze → Silver reconciliation
- Metadata preservation

### Status
- PASS = expectation met
- WARN = known issue requiring review
- FAIL = blocking DQ issue
- INFO = measurement only

In [0]:
%sql

DECLARE OR REPLACE VARIABLE dq_run_id STRING;

SET VARIABLE dq_run_id = uuid();

SELECT
    dq_run_id AS run_id,
    current_timestamp() AS executed_at;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS `ftw-week-08`.`01-control`.taxi_zones_silver_dq_results (

    run_id STRING,
    executed_at TIMESTAMP,

    layer STRING,
    dataset STRING,

    check_name STRING,
    check_type STRING,

    status STRING,
    severity STRING,

    fail_count BIGINT,
    total_count BIGINT,

    fail_pct DOUBLE,
    threshold_pct DOUBLE,

    metric_value DOUBLE,

    owner STRING,
    details STRING
)

COMMENT 'Silver Taxi Zones data quality results. One row per check per validation run.';

In [0]:
%sql

DELETE FROM `ftw-week-08`.`01-control`.taxi_zones_silver_dq_results

WHERE run_id = dq_run_id
  AND layer = '03-silver'
  AND dataset = 'taxi_zones';


INSERT INTO `ftw-week-08`.`01-control`.taxi_zones_silver_dq_results

WITH checks AS (

    SELECT
        COUNT(*) AS total_count,

        COUNT_IF(location_id IS NULL) AS location_id_nulls,

        COUNT_IF(
            borough IS NULL
            OR trim(borough) = ''
        ) AS borough_nulls,

        COUNT_IF(
            zone_name IS NULL
            OR trim(zone_name) = ''
        ) AS zone_name_nulls,

        COUNT_IF(
            service_zone IS NULL
            OR trim(service_zone) = ''
        ) AS service_zone_nulls,

        COUNT_IF(
            zone_classification IS NULL
            OR trim(zone_classification) = ''
        ) AS classification_nulls,

        COUNT_IF(
            location_id <= 0
        ) AS invalid_location_ids

    FROM `ftw-week-08`.`03-silver`.taxi_zones_clean
),

results AS (

    SELECT
        'location_id_not_null' AS check_name,
        'NOT_NULL' AS check_type,
        location_id_nulls AS fail_count,
        total_count,
        'HIGH' AS severity,
        'Silver location_id must not be NULL.' AS details

    FROM checks

    UNION ALL

    SELECT
        'borough_not_null',
        'NOT_NULL',
        borough_nulls,
        total_count,
        'HIGH',
        'Silver borough must not be NULL or blank.'

    FROM checks

    UNION ALL

    SELECT
        'zone_name_not_null',
        'NOT_NULL',
        zone_name_nulls,
        total_count,
        'HIGH',
        'Silver zone_name must not be NULL or blank.'

    FROM checks

    UNION ALL

    SELECT
        'service_zone_not_null',
        'NOT_NULL',
        service_zone_nulls,
        total_count,
        'HIGH',
        'Silver service_zone must not be NULL or blank.'

    FROM checks

    UNION ALL

    SELECT
        'zone_classification_not_null',
        'NOT_NULL',
        classification_nulls,
        total_count,
        'HIGH',
        'Silver zone_classification must not be NULL or blank.'

    FROM checks

    UNION ALL

    SELECT
        'location_id_positive',
        'RANGE',
        invalid_location_ids,
        total_count,
        'HIGH',
        'location_id must be greater than zero.'

    FROM checks
)

SELECT

    dq_run_id AS run_id,
    current_timestamp() AS executed_at,

    '03-silver' AS layer,
    'taxi_zones' AS dataset,

    check_name,
    check_type,

    CASE
        WHEN fail_count = 0 THEN 'PASS'
        ELSE 'FAIL'
    END AS status,

    severity,

    fail_count,
    total_count,

    ROUND(
        100.0 * fail_count / NULLIF(total_count, 0),
        2
    ) AS fail_pct,

    0.0 AS threshold_pct,

    CAST(fail_count AS DOUBLE) AS metric_value,

    'Data Engineering' AS owner,

    details

FROM results;

In [0]:
%sql

INSERT INTO `ftw-week-08`.`01-control`.taxi_zones_silver_dq_results

WITH duplicates AS (

    SELECT
        location_id,
        COUNT(*) AS cnt

    FROM `ftw-week-08`.`03-silver`.taxi_zones_clean

    GROUP BY location_id

    HAVING COUNT(*) > 1
),

counts AS (

    SELECT

        COALESCE(
            SUM(cnt - 1),
            0
        ) AS duplicate_rows,

        (
            SELECT COUNT(*)
            FROM `ftw-week-08`.`03-silver`.taxi_zones_clean
        ) AS total_rows

    FROM duplicates
)

SELECT

    dq_run_id AS run_id,
    current_timestamp() AS executed_at,

    '03-silver',
    'taxi_zones',

    'location_id_unique',
    'UNIQUENESS',

    CASE
        WHEN duplicate_rows = 0 THEN 'PASS'
        ELSE 'FAIL'
    END,

    'HIGH',

    duplicate_rows,
    total_rows,

    ROUND(
        100.0 * duplicate_rows
        / NULLIF(total_rows, 0),
        2
    ),

    0.0,

    CAST(duplicate_rows AS DOUBLE),

    'Data Engineering',

    'Each location_id should occur only once in Silver.'

FROM counts;

In [0]:
%sql

INSERT INTO `ftw-week-08`.`01-control`.taxi_zones_silver_dq_results

WITH checks AS (

    SELECT

        COUNT(*) AS total_count,

        COUNT_IF(
            service_zone != lower(trim(service_zone))
            OR service_zone = 'N/A'
        ) AS invalid_service_zone,

        COUNT_IF(
            borough != trim(borough)
        ) AS untrimmed_borough,

        COUNT_IF(
            zone_name != trim(zone_name)
        ) AS untrimmed_zone_name

    FROM `ftw-week-08`.`03-silver`.taxi_zones_clean
),

results AS (

    SELECT
        'service_zone_standardization' AS check_name,
        'TRANSFORMATION' AS check_type,
        invalid_service_zone AS fail_count,
        total_count,
        'HIGH' AS severity,
        'service_zone should be lowercase and source N/A should be converted to na.' AS details

    FROM checks

    UNION ALL

    SELECT
        'borough_trimmed' AS check_name,
        'TRANSFORMATION' AS check_type,
        untrimmed_borough AS fail_count,
        total_count,
        'MEDIUM' AS severity,
        'borough values should be trimmed.' AS details

    FROM checks

    UNION ALL

    SELECT
        'zone_name_trimmed' AS check_name,
        'TRANSFORMATION' AS check_type,
        untrimmed_zone_name AS fail_count,
        total_count,
        'MEDIUM' AS severity,
        'zone_name values should be trimmed.' AS details

    FROM checks
)

SELECT

    dq_run_id AS run_id,
    current_timestamp() AS executed_at,

    '03-silver' AS layer,
    'taxi_zones' AS dataset,

    check_name,
    check_type,

    CASE
        WHEN fail_count = 0 THEN 'PASS'
        ELSE 'FAIL'
    END AS status,

    severity,

    fail_count,
    total_count,

    ROUND(
        100.0 * fail_count
        / NULLIF(total_count, 0),
        2
    ) AS fail_pct,

    0.0 AS threshold_pct,

    CAST(fail_count AS DOUBLE) AS metric_value,

    'Data Engineering' AS owner,

    details

FROM results;

In [0]:
%sql

INSERT INTO `ftw-week-08`.`01-control`.taxi_zones_silver_dq_results

WITH checks AS (

    SELECT

        COUNT(*) AS total_count,

        COUNT_IF(
            zone_classification NOT IN (
                'unknown',
                'outside_nyc',
                'ewr',
                'nyc_borough',
                'other_special'
            )
        ) AS invalid_domain,

        COUNT_IF(
            (location_id = 264
                AND zone_classification != 'unknown')

            OR

            (location_id = 265
                AND zone_classification != 'outside_nyc')

            OR

            (lower(trim(borough)) = 'ewr'
                AND zone_classification != 'ewr')
        ) AS invalid_special,

        COUNT_IF(
            lower(trim(borough)) IN (
                'bronx',
                'brooklyn',
                'manhattan',
                'queens',
                'staten island'
            )
            AND zone_classification != 'nyc_borough'
        ) AS invalid_borough_classification,

        COUNT_IF(
            lower(trim(borough)) NOT IN (
                'bronx',
                'brooklyn',
                'manhattan',
                'queens',
                'staten island',
                'ewr'
            )
            AND location_id NOT IN (264, 265)
            AND zone_classification != 'other_special'
        ) AS invalid_other_special

    FROM `ftw-week-08`.`03-silver`.taxi_zones_clean
),

results AS (

    SELECT
        'zone_classification_domain' AS check_name,
        'DOMAIN' AS check_type,
        invalid_domain AS fail_count,
        total_count,
        'HIGH' AS severity,
        'zone_classification must use the approved classification values.' AS details

    FROM checks

    UNION ALL

    SELECT
        'special_location_classification' AS check_name,
        'CONSISTENCY' AS check_type,
        invalid_special AS fail_count,
        total_count,
        'HIGH' AS severity,
        'LocationID 264, LocationID 265, and EWR must follow the special classification rules.' AS details

    FROM checks

    UNION ALL

    SELECT
        'nyc_borough_classification' AS check_name,
        'CONSISTENCY' AS check_type,
        invalid_borough_classification AS fail_count,
        total_count,
        'HIGH' AS severity,
        'The five NYC boroughs should be classified as nyc_borough.' AS details

    FROM checks

    UNION ALL

    SELECT
        'other_special_classification' AS check_name,
        'CONSISTENCY' AS check_type,
        invalid_other_special AS fail_count,
        total_count,
        'MEDIUM' AS severity,
        'Non-standard borough values should be classified as other_special.' AS details

    FROM checks
)

SELECT

    dq_run_id AS run_id,
    current_timestamp() AS executed_at,

    '03-silver' AS layer,
    'taxi_zones' AS dataset,

    check_name,
    check_type,

    CASE
        WHEN fail_count = 0 THEN 'PASS'
        ELSE 'FAIL'
    END AS status,

    severity,

    fail_count,
    total_count,

    ROUND(
        100.0 * fail_count
        / NULLIF(total_count, 0),
        2
    ) AS fail_pct,

    0.0 AS threshold_pct,

    CAST(fail_count AS DOUBLE) AS metric_value,

    'Data Engineering' AS owner,

    details

FROM results;

In [0]:
%sql

INSERT INTO `ftw-week-08`.`01-control`.taxi_zones_silver_dq_results

WITH counts AS (

    SELECT

        (
            SELECT COUNT(*)
            FROM `ftw-week-08`.`02-bronze`.taxi_zones_raw
        ) AS bronze_count,

        (
            SELECT COUNT(*)
            FROM `ftw-week-08`.`03-silver`.taxi_zones_clean
        ) AS silver_count
)

SELECT

    dq_run_id AS run_id,
    current_timestamp() AS executed_at,

    '03-silver',
    'taxi_zones',

    'bronze_silver_row_count_reconciliation',
    'RECONCILIATION',

    CASE
        WHEN bronze_count = silver_count THEN 'PASS'
        ELSE 'FAIL'
    END,

    'HIGH',

    ABS(bronze_count - silver_count),
    bronze_count,

    ROUND(
        100.0 * ABS(bronze_count - silver_count)
        / NULLIF(bronze_count, 0),
        2
    ),

    0.0,

    CAST(silver_count AS DOUBLE),

    'Data Engineering',

    CONCAT(
        'Bronze rows = ',
        bronze_count,
        '; Silver rows = ',
        silver_count
    )

FROM counts;

In [0]:
%sql

INSERT INTO `ftw-week-08`.`01-control`.taxi_zones_silver_dq_results

WITH missing_in_silver AS (

    SELECT
        CAST(b.location_id AS INT) AS location_id

    FROM `ftw-week-08`.`02-bronze`.taxi_zones_raw b

    LEFT ANTI JOIN
        `ftw-week-08`.`03-silver`.taxi_zones_clean s

    ON CAST(b.location_id AS INT) = s.location_id
),

extra_in_silver AS (

    SELECT
        s.location_id

    FROM `ftw-week-08`.`03-silver`.taxi_zones_clean s

    LEFT ANTI JOIN
        `ftw-week-08`.`02-bronze`.taxi_zones_raw b

    ON CAST(b.location_id AS INT) = s.location_id
),

counts AS (

    SELECT

        (SELECT COUNT(*)
         FROM missing_in_silver) AS missing_count,

        (SELECT COUNT(*)
         FROM extra_in_silver) AS extra_count
)

SELECT

    dq_run_id AS run_id,
    current_timestamp() AS executed_at,

    '03-silver',
    'taxi_zones',

    'bronze_silver_location_id_reconciliation',
    'RECONCILIATION',

    CASE
        WHEN missing_count = 0
         AND extra_count = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END,

    'HIGH',

    missing_count + extra_count,

    (
        SELECT COUNT(*)
        FROM `ftw-week-08`.`02-bronze`.taxi_zones_raw
    ),

    ROUND(
        100.0 * (missing_count + extra_count)
        / NULLIF(
            (
                SELECT COUNT(*)
                FROM `ftw-week-08`.`02-bronze`.taxi_zones_raw
            ),
            0
        ),
        2
    ),

    0.0,

    CAST(
        missing_count + extra_count
        AS DOUBLE
    ),

    'Data Engineering',

    CONCAT(
        'Missing in Silver = ',
        missing_count,
        '; Extra in Silver = ',
        extra_count
    )

FROM counts;

In [0]:
%sql

INSERT INTO `ftw-week-08`.`01-control`.taxi_zones_silver_dq_results

SELECT

    dq_run_id AS run_id,
    current_timestamp() AS executed_at,

    '03-silver',
    'taxi_zones',

    'source_metadata_not_null',
    'METADATA',

    CASE
        WHEN COUNT_IF(
            source_file IS NULL
            OR trim(source_file) = ''

            OR batch_id IS NULL
            OR trim(batch_id) = ''

            OR ingested_at IS NULL
        ) = 0

        THEN 'PASS'
        ELSE 'FAIL'
    END,

    'MEDIUM',

    COUNT_IF(
        source_file IS NULL
        OR trim(source_file) = ''

        OR batch_id IS NULL
        OR trim(batch_id) = ''

        OR ingested_at IS NULL
    ),

    COUNT(*),

    ROUND(
        100.0 *
        COUNT_IF(
            source_file IS NULL
            OR trim(source_file) = ''

            OR batch_id IS NULL
            OR trim(batch_id) = ''

            OR ingested_at IS NULL
        )
        / NULLIF(COUNT(*), 0),
        2
    ),

    0.0,

    CAST(
        COUNT_IF(
            source_file IS NULL
            OR trim(source_file) = ''

            OR batch_id IS NULL
            OR trim(batch_id) = ''

            OR ingested_at IS NULL
        )
        AS DOUBLE
    ),

    'Data Engineering',

    'Bronze source metadata should be preserved in Silver.'

FROM `ftw-week-08`.`03-silver`.taxi_zones_clean;

In [0]:
%sql

SELECT

    check_name,
    check_type,
    status,
    severity,
    fail_count,
    total_count,
    fail_pct,
    threshold_pct,
    metric_value,
    details

FROM `ftw-week-08`.`01-control`.taxi_zones_silver_dq_results

WHERE run_id = dq_run_id

ORDER BY

    CASE status
        WHEN 'FAIL' THEN 1
        WHEN 'WARN' THEN 2
        WHEN 'PASS' THEN 3
        WHEN 'INFO' THEN 4
        ELSE 5
    END,

    check_name;

In [0]:
%sql

SELECT

    dq_run_id AS run_id,

    CASE
        WHEN COUNT_IF(status = 'FAIL') = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS silver_exit_gate,

    COUNT_IF(status = 'PASS') AS pass_count,

    COUNT_IF(status = 'WARN') AS warn_count,

    COUNT_IF(status = 'FAIL') AS fail_count,

    COUNT_IF(status = 'INFO') AS info_count

FROM `ftw-week-08`.`01-control`.taxi_zones_silver_dq_results

WHERE run_id = dq_run_id;

In [0]:
%sql

SELECT
    borough,
    zone_classification,
    COUNT(*) AS row_count
FROM `ftw-week-08`.`03-silver`.taxi_zones_clean
GROUP BY
    borough,
    zone_classification
ORDER BY
    row_count DESC;